In [0]:

# Load the cleaned Silver Delta table
silver_df = spark.table("workspace.default.books_silver")

# Verify the dataset
print("Total books:", silver_df.count())

print("Silver table schema:")
silver_df.printSchema()

display(silver_df.limit(10))


In [0]:

from pyspark.sql.functions import col, count

# Group books by publication year
books_by_year = (
    silver_df
    .filter(col("first_publish_year").isNotNull())
    .groupBy("first_publish_year")
    .agg(count("*").alias("total_books"))
    .orderBy("first_publish_year")
)

# Save the Gold table
(
    books_by_year.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.books_gold_by_year")
)

# Verify the result
print("Gold table created: books_gold_by_year")
display(books_by_year)


In [0]:

from pyspark.sql.functions import explode, trim, col, count

# Expand the authors array into individual rows
authors_df = (
    silver_df
    .select(
        "work_id",
        explode("authors").alias("author")
    )
    .filter(col("author").isNotNull())
    .filter(trim(col("author")) != "")
)

# Count books for each author
top_authors = (
    authors_df
    .groupBy("author")
    .agg(count("work_id").alias("total_books"))
    .orderBy(col("total_books").desc(), col("author"))
)

# Save Gold table
(
    top_authors.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.books_gold_by_author")
)

print("Gold table created: books_gold_by_author")
display(top_authors.limit(10))


In [0]:

from pyspark.sql.functions import floor, col, count

# Calculate publication decade
books_by_decade = (
    silver_df
    .filter(col("first_publish_year").isNotNull())
    .withColumn(
        "publication_decade",
        (floor(col("first_publish_year") / 10) * 10).cast("int")
    )
    .groupBy("publication_decade")
    .agg(count("*").alias("total_books"))
    .orderBy("publication_decade")
)

# Save Gold table
(
    books_by_decade.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.books_gold_by_decade")
)

print("Gold table created: books_gold_by_decade")
display(books_by_decade)


In [0]:

from pyspark.sql.functions import col, desc

# Select books with valid edition counts
books_by_editions = (
    silver_df
    .filter(col("edition_count").isNotNull())
    .filter(col("edition_count") > 0)
    .select(
        "work_id",
        "title",
        "authors",
        "first_publish_year",
        "edition_count"
    )
    .orderBy(desc("edition_count"), col("title"))
)

# Save Gold table
(
    books_by_editions.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.books_gold_by_editions")
)

print("Gold table created: books_gold_by_editions")
display(books_by_editions.limit(10))


In [0]:

gold_tables = [
    "books_gold_by_year",
    "books_gold_by_author",
    "books_gold_by_decade",
    "books_gold_by_editions"
]

for table_name in gold_tables:
    df = spark.table(f"workspace.default.{table_name}")

    print(f"\nTable: {table_name}")
    print(f"Rows: {df.count()}")
    print("Columns:", df.columns)


In [0]:
# Data quality checks for Gold editions table

df = spark.table("workspace.default.books_gold_by_editions")

print("Total records:", df.count())

print("Null counts:")
df.select([
    __import__("pyspark").sql.functions.count(
        __import__("pyspark").sql.functions.when(
            __import__("pyspark").sql.functions.col(c).isNull(), c
        )
    ).alias(c)
    for c in df.columns
]).show()

print("Duplicate work IDs:")
df.groupBy("work_id").count().filter("count > 1").show()

print("Invalid publication years:")
df.filter(
    (df.first_publish_year < 0) |
    (df.first_publish_year > 2026)
).show()

print("Negative edition counts:")
df.filter(df.edition_count < 0).show()

In [0]:

from pyspark.sql.functions import col

df = spark.table("workspace.default.books_gold_by_editions")

# Critical field checks
missing_required = df.filter(
    col("work_id").isNull() |
    col("title").isNull()
).count()

# Duplicate work IDs
duplicate_ids = (
    df.groupBy("work_id")
      .count()
      .filter(col("count") > 1)
      .count()
)

# Invalid publication years (NULL is allowed)
invalid_years = df.filter(
    col("first_publish_year").isNotNull() &
    (
        (col("first_publish_year") < 0) |
        (col("first_publish_year") > 2026)
    )
).count()

# Negative edition counts (NULL is allowed)
negative_editions = df.filter(
    col("edition_count").isNotNull() &
    (col("edition_count") < 0)
).count()

print("DATA QUALITY REPORT")
print("Total records:", df.count())
print("Missing required fields:", missing_required)
print("Duplicate work IDs:", duplicate_ids)
print("Invalid publication years:", invalid_years)
print("Negative edition counts:", negative_editions)

if any([
    missing_required,
    duplicate_ids,
    invalid_years,
    negative_editions
]):
    raise Exception("Gold data quality validation FAILED")

print("All critical data quality checks PASSED.")
